In [1]:
import h5py
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import umap
import pandas as pd
import numpy as np
from umap.umap_ import nearest_neighbors, fuzzy_simplicial_set, spectral_layout

In [3]:
def load_hdf5_features(hdf5_path, dataset_name, flatten=True):
    """
    Load features from HDF5 file and return concatenated data with labels.

    Args:
        hdf5_path (str or Path): Path to the HDF5 file
        dataset_name (str): Name of the dataset to read (e.g., 'layer4_features')
        max_groups (int, optional): Maximum number of groups to read. If None, read all groups
        flatten (bool): Whether to flatten features to 2D array (samples, features). Default: True

    Returns:
        data (np.ndarray): Concatenated feature data of shape (n_samples, n_features)
        group_labels (np.ndarray): Numeric labels for each sample
        group_names (list): Original group names in order
        le (LabelEncoder): Trained LabelEncoder for inverse transformation
        dic_feature (list): Original group names for each sample
    """
    hdf5_path = Path(hdf5_path)
    data_list = []
    dic_feature = []
    labels = []

    with h5py.File(hdf5_path, 'r') as file:
        group_names = list(file.keys())

        for i, group_name in enumerate(group_names):
            group = file[group_name]
            if dataset_name not in group:
                raise KeyError(f"Dataset '{dataset_name}' not found in group '{group_name}'")
            
            dataset = group[dataset_name][:]
            dic_feature.extend([group_name] * dataset.shape[0])
            
            if flatten:
                data_list.append(dataset.reshape(dataset.shape[0], -1))
            else:
                data_list.append(dataset) 
            
            labels.append([i] * dataset.shape[0])

    # Concatenate all data
    data = np.concatenate(data_list, axis=0)
    labels = np.concatenate(labels, axis=0)

    # Encode group names to numeric labels 
    le = LabelEncoder()
    group_labels = le.fit_transform(dic_feature)

    return data, group_labels, group_names, le, dic_feature

# 使用示例
hdf5_filename = r"E:\workspace\Neural networks\extract_feature\gwas\13features_trained_resnet\resnet\feature.hdf5"
data, group_labels, group_names, le, dic_feature = load_hdf5_features(hdf5_filename, dataset_name='layer4_features')

In [5]:
def find_best_umap_dim(data, max_dim=40, n_neighbors=5, min_dist=0.05, metric='cosine',
                       epsilon=0.01, window_size=3, random_state=42):
    """
    Find the optimal UMAP dimensionality by calculating variance ratios.
    Checks conditions incrementally and stops early when criteria are met.

    Args:
        data (np.ndarray): Original feature matrix of shape (n_samples, n_features)
        max_dim (int): Maximum dimensionality to test
        n_neighbors (int): UMAP n_neighbors parameter
        min_dist (float): UMAP min_dist parameter
        metric (str): UMAP distance metric
        epsilon (float): Variance ratio threshold for early stopping
        window_size (int): Number of consecutive dimensions to check for low variance
        random_state (int): Random seed for reproducibility

    Returns:
        best_dim (int): Optimal dimensionality found
    """

    for d in range(1, max_dim + 1):
        reducer = umap.UMAP(
            n_components=d,
            random_state=random_state,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            metric=metric
        )
        X_umap = reducer.fit_transform(data)
        print(X_umap)
        var_d = np.var(X_umap, axis=0)  # Variance of each dimension
        total_var = np.sum(var_d)
        ratio = var_d / total_var
        print(ratio)
        
        # Start checking from the 3rd dimension onward
        if d >= window_size:
            last_ratios = ratio[-window_size:]
            print(last_ratios)
            # Check if the last dimension's variance ratio is below epsilon for all recent dimensions
            if np.all(np.array(last_ratios) < epsilon):
                return d
    return max_dim

best_dim = find_best_umap_dim(data, max_dim=40)
print("Optimal dimensionality:", best_dim)

E:\software\conda\envs\py38\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


[[-10.448616 ]
 [-10.428548 ]
 [ -3.6424348]
 ...
 [ -4.6194015]
 [ -4.6156116]
 [ -4.4575105]]
[1.]


E:\software\conda\envs\py38\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


[[-0.10428458 -1.3657476 ]
 [ 0.00515117 -1.2435688 ]
 [ 1.8006042  -1.686656  ]
 ...
 [-3.2063684   1.71875   ]
 [-3.1519964   1.7657753 ]
 [-3.1885629   1.7624964 ]]
[0.626146   0.37385392]


E:\software\conda\envs\py38\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


...

[[1.3466696  0.10604472 3.4255445  ... 5.0146685  4.199556   8.153417  ]
 [1.4457073  0.14945638 3.4167514  ... 4.9990435  4.243505   8.126125  ]
 [2.0030208  0.857857   3.0715885  ... 4.7834635  4.4705315  7.917627  ]
 ...
 [0.5530434  1.5996011  3.535282   ... 5.531365   4.4564023  7.8973336 ]
 [0.5597666  1.6145376  3.5310822  ... 5.531417   4.467365   7.9002776 ]
 [0.5430641  1.6188933  3.53206    ... 5.5366187  4.4745636  7.904973  ]]
[0.565793   0.2079438  0.05092765 0.05962018 0.02331291 0.03121983
 0.0220794  0.00626836 0.01155292 0.01521341 0.00606849]
[0.01155292 0.01521341 0.00606849]


E:\software\conda\envs\py38\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


[[1.5126692  0.11311481 3.4897523  ... 4.4157624  8.407156   3.9452853 ]
 [1.6055377  0.15478976 3.457403   ... 4.4273176  8.368829   3.884732  ]
 [1.9658089  0.7960282  3.0194533  ... 4.4194307  7.9756575  3.8413444 ]
 ...
 [0.7112834  1.5119181  3.5533245  ... 4.4373984  7.832895   4.347417  ]
 [0.728237   1.5278522  3.5473695  ... 4.4437604  7.82713    4.34617   ]
 [0.700398   1.5201161  3.5642083  ... 4.436202   7.8300757  4.3626604 ]]
[0.56973934 0.20090394 0.05246157 0.05938028 0.02211947 0.03096929
 0.02012909 0.00565969 0.01079231 0.015737   0.00479018 0.00731791]
[0.015737   0.00479018 0.00731791]


E:\software\conda\envs\py38\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


[[1.6197019  0.18976142 3.297678   ... 8.232793   5.994773   8.436191  ]
 [1.681026   0.20880413 3.324044   ... 8.235388   5.970062   8.421843  ]
 [2.07329    0.84371054 3.0621183  ... 8.021261   6.166063   8.580094  ]
 ...
 [0.7332891  1.4920352  3.4973073  ... 7.7056828  5.779027   8.295596  ]
 [0.7349907  1.5051254  3.5100806  ... 7.6933165  5.7657104  8.282694  ]
 [0.6900833  1.4908215  3.5179412  ... 7.694256   5.7583914  8.281501  ]]
[0.56669116 0.20582595 0.05245818 0.05977901 0.02257876 0.02830246
 0.01953129 0.00563197 0.01134575 0.01276435 0.00509117 0.00787762
 0.00212219]
[0.00509117 0.00787762 0.00212219]
Optimal dimensionality: 13


In [ ]:
def umap_spectral_init(data, n_neighbors=5, metric="cosine", dim=32, random_state=42):
    """
    Use UMAP's internal functions to manually compute spectral embedding initialization.
    
    Args:
        data (np.ndarray): Input feature matrix, shape (number of samples, number of features)
        n_neighbors (int): Number of neighbors for kNN graph
        metric (str): Distance metric (e.g., "euclidean", "cosine")
        dim (int): Target embedding dimension
        random_state (int or np.random.RandomState): Random seed

    Returns:
        embedding (np.ndarray): Spectral initialized embedding coordinates, shape (number of samples, dim)
        graph (scipy.sparse.csr_matrix): Fuzzy adjacency graph
        knn_indices (np.ndarray): kNN indices
        knn_dists (np.ndarray): kNN distances
    """
    if isinstance(random_state, int):
        random_state = np.random.RandomState(random_state)

    # Construct kNN graph
    knn_indices, knn_dists, rp_forest = nearest_neighbors(
        X=data,
        n_neighbors=n_neighbors,
        random_state=random_state,
        metric=metric,
        metric_kwds={},
        angular=False
    )

    # Fuzzy adjacency matrix
    graph, sigmas, rhos = fuzzy_simplicial_set(
        X=data,
        n_neighbors=n_neighbors,
        random_state=random_state,
        metric=metric,
        knn_indices=knn_indices,
        knn_dists=knn_dists,
        set_op_mix_ratio=1.0,
        local_connectivity=1.0
    )

    # Spectral layout computation for initial embedding
    embedding = spectral_layout(
        data,
        graph,
        dim=dim,
        random_state=random_state
    )

    return embedding, graph, knn_indices, knn_dists


embedding_Laplacian, graph, knn_indices, knn_dists = umap_spectral_init(data, n_neighbors=5, metric="cosine", dim=best_dim)
print("Embedding shape:", embedding_Laplacian.shape)


In [5]:
best_dim = 13
reducer_use = umap.UMAP(
    n_components=best_dim, 
    random_state=42,
    n_neighbors=5,
    min_dist=0.05,
    metric='cosine',
)
embedding = reducer_use.fit_transform(data)

E:\software\conda\envs\py38\lib\site-packages\umap\umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [6]:
def detect_outliers_std(data):
    """
    Detect outliers using standard deviation method (Z-score).
    
    Args:
        data (list of arrays or np.ndarray): Input data samples
        
    Returns:
        list: Boolean list indicating outliers (True) and inliers (False)
    """
    # Convert list to ndarray for batch computation
    data = np.vstack(data)

    # Calculate mean and standard deviation
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)

    # Avoid division by zero
    std[std == 0] = 1

    # Calculate Z-scores for each sample
    z_scores = np.abs((data - mean) / std)

    # Define threshold (2 standard deviations)
    threshold = 2

    # Identify outliers: samples exceeding threshold in any dimension
    outliers = np.any(z_scores > threshold, axis=1)

    return outliers.tolist()


def detect_outliers(data, method=None):
    """
    Detect outliers using Z-score method with optional additional processing.
    
    Args:
        data (list of arrays or np.ndarray): Input data samples
        method (optional): Additional outlier detection method (not fully implemented)
        
    Returns:
        list: Boolean list indicating outliers (True) and inliers (False)
    """
    # Convert list to ndarray for batch computation
    data = np.vstack(data)

    # Calculate mean and standard deviation
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)

    # Avoid division by zero
    std[std == 0] = 1

    # Calculate Z-scores for each sample
    z_scores = np.abs((data - mean) / std)

    # Define threshold (1 standard deviation)
    threshold = 1

    # Identify outliers: samples exceeding threshold in any dimension
    outliers = np.any(z_scores > threshold, axis=1)
    
    # Additional processing if method is specified (incomplete implementation)
    if method is not None:
        for j in range(data.shape[1]):
            mean_co = np.mean(data[:, j], axis=0)
            std_co = np.std(data[:, j], axis=0)
            z_scores_co = np.abs((data.shape[1] - mean_co) / std_co)
            threshold_co = 1.5
            if z_scores_co > threshold_co:
                outliers[j] = True
                
    return outliers.tolist()

In [ ]:
def process_and_save(csv_filename, csv_mean_filename):
    """
    Process embedding data by grouping, detecting outliers, and saving results to CSV files.
    
    Args:
        csv_filename (str): Path to save the full dataset with outlier flags
        csv_mean_filename (str): Path to save the group means (non-outlier data)
    """
    now = ''
    fold_data = []
    fold_now = []
    
    # Group embedding data by dic_feature values
    for i in range(len(embedding)):
        if i == 0:
            now = dic_feature[i]
            fold_now.append(embedding[i])
        elif dic_feature[i] != now:
            fold_data.append(fold_now)
            fold_now = [embedding[i]]
            now = dic_feature[i]
        elif dic_feature[i] == now:
            fold_now.append(embedding[i])
    
    fold_data.append(fold_now)
    
    new_fold_data = []
    new_dic_features = []
    is_outlier_flags = []  # List to record outlier status for each data point
    
    # Process each group to detect outliers
    for group_data, group_name in zip(fold_data, list(dict.fromkeys(dic_feature))):
        outliers = detect_outliers_std(group_data)
        print(outliers)
        # Record outlier status for each data point
        is_outlier_flags.extend(outliers)
        # Add group information for each data point
        new_dic_features.extend([group_name] * len(group_data))
        # Add original data points (including outliers)
        new_fold_data.extend(group_data)
    
    solitude_combined_features = np.vstack(new_fold_data)
    
    # Create DataFrame with group info and outlier flags
    df = pd.DataFrame({
        'Group': new_dic_features,  # Group information for each data point
        'Is_Outlier': is_outlier_flags,  # Outlier flag for each data point
    })
    
    # Add feature columns
    for i in range(solitude_combined_features.shape[1]):
        df[f'Feature_{i + 1}'] = solitude_combined_features[:, i]
        
    # Calculate group means excluding outliers
    group_means = {}
    for group_data, group_name in zip(fold_data, list(dict.fromkeys(dic_feature))):
        outliers = detect_outliers_std(group_data)
    
        # Keep only non-outlier data
        non_outlier_data = np.array(group_data)[~np.array(outliers)]
    
        if len(non_outlier_data) > 0:  # Ensure non-outlier data is not empty
            group_means[group_name] = np.mean(non_outlier_data, axis=0)
    
    # Convert means dictionary to DataFrame
    df_means = pd.DataFrame.from_dict(group_means, orient='index')
    
    # Rename columns
    df_means.columns = [f'Feature_{i + 1}' for i in range(df_means.shape[1])]
    df_means.index.name = 'Group'  # Set index name
    
    # Save DataFrames to CSV files
    df.to_csv(csv_filename, index=False)
    df_means.to_csv(csv_mean_filename)
    
# Define output file paths
csv_filename = r'E:\workspace\Neural networks\extract_feature\gwas\13features_trained_resnet\13features_trained_resnet_Laplacian.csv'
csv_mean_filename = r'E:\workspace\Neural networks\extract_feature\gwas\13features_trained_resnet\13features_trained_resnet_mean.csv'
process_and_save(csv_filename, csv_mean_filename)